# Aula 14B · Automação de equipamentos

**Noite B — o problema.** Nenhum conceito novo: esta noite é laboratório sobre o
[capítulo 14 do
site](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/),
que a [noite
A](https://colab.research.google.com/github/lacouth/python_telecom-site/blob/main/notebooks/aula14a-automacao.ipynb)
apresentou. Faltou à noite A? Cada exercício traz o link 📖 para a seção que ele usa
— e o caderno da noite A tem os passos.

**Roteiro** (1h40): 🔥 aquecimento · ⚠️ a regra da noite · 🎯 prática · 📟 resolvendo o
chamado · 📋 a Lista 14, começada aqui · 🚪 antes de sair

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🎯 Sua vez:** escreva a solução e rode a célula `confere` logo abaixo dela —
  ✅ quer dizer que acertou, ❌ mostra o que ainda falta. A dica está recolhida:
  tente antes de abrir.
- Travou? Antes da dica, abra o link 📖 do exercício: a seção do capítulo que
  ele usa tem o exemplo completo.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler agora (usa coisas que só veremos mais tarde).
import math


def _mostra(argumentos):
    return ", ".join(repr(a) for a in argumentos)


def _igual(veio, esperado):
    if isinstance(esperado, float) and isinstance(veio, (int, float)):
        return math.isclose(veio, esperado, abs_tol=1e-9)
    return veio == esperado


def confere(funcao, casos):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if _igual(veio, esperado):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if _igual(valor, esperado):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")


# --- dados desta aula ---
# simulador: equipamentos SSH da Maré Net — não precisa ler (faz o papel dos switches).
# Imita a interface da biblioteca Netmiko: ConnectHandler(...), send_command(...),
# disconnect() e as mesmas exceções. Com equipamentos de verdade, a única linha que
# muda é o import: from netmiko import ConnectHandler, ...
import time


class NetmikoTimeoutException(Exception):
    """O equipamento não respondeu a tempo."""


class NetmikoAuthenticationException(Exception):
    """Usuário ou senha recusados."""


_VERSAO = """Cisco IOS Software, C2960X Software (C2960X-UNIVERSALK9-M), Version {versao}, RELEASE SOFTWARE (fc3)
{nome} uptime is {uptime}
System image file is "flash:c2960x-universalk9-mz.{versao}.bin"
cisco WS-C2960X-48FPD-L (APM86XXX) processor with 524288K bytes of memory."""

_CABECALHO = "Interface              IP-Address      OK? Method Status                Protocol"

_EQUIPAMENTOS = {
    "10.0.1.20": {"nome": "SWITCH-CENTRO-01", "versao": "15.2(7)E7", "uptime": "3 weeks, 2 days, 4 hours",
                  "interfaces": [("Vlan10", "10.0.1.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/3", "unassigned", "down", "down")]},
    "10.0.2.20": {"nome": "SWITCH-NORTE-02", "versao": "15.2(7)E7", "uptime": "12 days, 7 hours",
                  "interfaces": [("Vlan20", "10.0.2.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "administratively down", "down"),
                                 ("GigabitEthernet1/0/3", "unassigned", "down", "down"),
                                 ("GigabitEthernet1/0/4", "unassigned", "down", "down")]},
    "10.0.3.20": {"nome": "SWITCH-SUL-03", "falha": "tempo"},
    "10.0.4.20": {"nome": "SWITCH-LESTE-04", "falha": "senha"},
    "10.0.5.20": {"nome": "SWITCH-OESTE-05", "versao": "15.2(4)E10", "uptime": "1 year, 5 weeks",
                  "interfaces": [("Vlan50", "10.0.5.20", "up", "up"),
                                 ("GigabitEthernet1/0/1", "unassigned", "up", "up"),
                                 ("GigabitEthernet1/0/2", "unassigned", "up", "up")]},
}
_SENHA = "marenet"


class _Conexao:
    def __init__(self, dados):
        self._dados = dados

    def send_command(self, comando):
        comando = " ".join(comando.split())
        if comando == "show version":
            return _VERSAO.format(**self._dados)
        if comando == "show ip interface brief":
            linhas = [_CABECALHO]
            for nome, ip, estado, protocolo in self._dados["interfaces"]:
                linhas.append(f"{nome:<23}{ip:<16}YES manual {estado:<22}{protocolo}")
            return "\n".join(linhas)
        if comando == "show clock":
            return "*14:03:17.123 BRT Mon Mar 2 2026"
        return "% Invalid input detected at '^' marker."

    def disconnect(self):
        pass

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.disconnect()


def ConnectHandler(device_type, host, username, password, timeout=5, **extras):
    """Abre uma "sessão SSH" com o equipamento simulado."""
    dados = _EQUIPAMENTOS.get(host)
    if dados is None or dados.get("falha") == "tempo":
        time.sleep(0.2)
        raise NetmikoTimeoutException(f"TCP connection to device failed: {host}")
    if dados.get("falha") == "senha" or password != _SENHA:
        raise NetmikoAuthenticationException(f"Authentication to device failed: {host}")
    return _Conexao(dados)


print("equipamentos simulados prontos (senha do usuário noc: marenet)")

## 🔥 Aquecimento — a pergunta que ficou

No fim da noite A ficou uma pergunta sem resposta. Responda de cabeça, sem rodar.

O `send_command("show version")` devolve:
a) um dicionário com os campos  b) a saída do comando, como texto  c) `True` se deu
certo  d) uma lista de linhas

<details>
<summary><b>Resposta da 1</b></summary>

**b**. Transformar o texto em dado é trabalho do script.

</details>

## ⚠️ A regra da noite

**Quem falha é registrado, e a volta continua.** Cinquenta equipamentos, um `try` **por equipamento**, dentro do laço. O
`except` anota quem não respondeu e por quê, e o laço segue para o próximo.

E a saída do comando é **texto**: quem a transforma em dado é o parser da
Aula 09 — `splitlines`, `partition`, `split(maxsplit=...)` —, testado também com a
linha que não é a esperada.

## 🎯 Prática

Nenhum passo novo: cada exercício usa uma seção da noite A, indicada no link 📖
logo acima dele. Do mais simples para o mais completo.

Vem do bloco *4. O coletor completo* da noite A, e os passos citados na dica são de lá: 📖 [capítulo 14 · O coletor completo](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/#o-coletor-completo)

### 🎯 Sua vez — Quem responde?

Escreva `alcancaveis(hosts)`, que devolve a lista dos hosts em que **foi possível
conectar** (na ordem). Use `try` com os dois `except`.

In [ ]:
def alcancaveis(hosts):
    # sua solução aqui
    pass

In [ ]:
confere(alcancaveis, [
    ((["10.0.1.20", "10.0.3.20", "10.0.4.20", "10.0.5.20"],), ["10.0.1.20", "10.0.5.20"]),
    ((["10.0.3.20"],), []),
])

<details>
<summary><b>💡 Dica</b></summary>

Para cada host: `try:` com o `with ConnectHandler(...)` e, dentro dele, o `append`; os
dois `except` só passam para o próximo (`pass` ou `continue`).

</details>

## 📟 Resolvendo o chamado

O chamado que a noite A abriu:

> **Chamado #1609 — NOC Maré Net**
>
> *"Estagiário, vamos atualizar o firmware dos switches para a versão **15.2(7)E7**.
> Preciso saber **quais estão em outra versão** e **quais nem responderam** — esses
> vão para a equipe de campo. Hoje alguém entra em um por um pelo terminal e anota
> numa planilha."*

### 🎯 Sua vez — A auditoria de firmware

Escreva `auditoria_firmware(hosts, versao_alvo)`, que devolve **dois valores**, cada um
uma lista de hosts na ordem: os que responderam **numa versão diferente** de
`versao_alvo`, e os que **não puderam ser consultados** (tempo esgotado ou senha
recusada). A função `versao_de` já vem escrita no esqueleto.

In [ ]:
def versao_de(saida):
    for linha in saida.splitlines():
        if "Version " in linha:
            _, _, depois = linha.partition("Version ")
            versao, _, _ = depois.partition(",")
            return versao
    return None


def auditoria_firmware(hosts, versao_alvo):
    # sua solução aqui
    pass

In [ ]:
confere(auditoria_firmware, [
    ((["10.0.1.20", "10.0.2.20", "10.0.3.20", "10.0.4.20", "10.0.5.20"], "15.2(7)E7"),
     (["10.0.5.20"], ["10.0.3.20", "10.0.4.20"])),
    ((["10.0.1.20"], "15.2(7)E7"), ([], [])),
])

<details>
<summary><b>💡 Dica</b></summary>

Duas listas vazias antes do laço. Para cada host, um `try` com o `with`, a versão e,
**se** ela for diferente da alvo, o `append` na primeira lista; os dois `except` fazem o
`append` na segunda.

</details>

**Resposta ao chamado:** só o `10.0.5.20` (SWITCH-OESTE-05, na `15.2(4)E10`) precisa de
atualização entre os que responderam. O `10.0.3.20` não respondeu e o `10.0.4.20`
recusou a senha — esses dois vão para a equipe de campo, com o motivo de cada um. E da
próxima vez, para cinquenta switches, o script leva o mesmo tempo que você levaria para
abrir o primeiro terminal.

## 📋 A lista, começada aqui

Abra a [Lista 14](https://lacouth.github.io/python_telecom-site/listas/lista14/) —
ou direto o [caderno dela no
Colab](https://colab.research.google.com/github/lacouth/python_telecom-site/blob/main/notebooks/lista14.ipynb)
— e rode a célula de preparo. O **exercício 1** a turma faz junto, respondendo às
perguntas abaixo **antes** de escrever; do 2 em diante, cada um no seu ritmo, com o
professor circulando.

**Exercício 1 — `versao_de`**

**1.** A `versao_de` veio pronta no esqueleto do chamado. Sem olhar para ela: onde está a versão?

<details>
<summary><b>Resposta da 1</b></summary>

Na linha que contém `"Version "`. Percorra `saida.splitlines()` e pare nela. Escrever de novo o que você acabou de usar pronto é o que fixa.

</details>

**2.** Como cortar a versão da linha?

<details>
<summary><b>Resposta da 2</b></summary>

Dois `partition`: o primeiro, em `"Version "`, fica com o que vem **depois**; o segundo, em `","`, fica com o que vem **antes**.

</details>

**3.** Como conferir?

<details>
<summary><b>Resposta da 3</b></summary>

Com a saída do enunciado, a resposta é `"15.2(7)E7"` — sem espaço antes e sem a vírgula depois.

</details>

## 🚪 Antes de sair

Responda de cabeça, sem rodar.

**1.** `"Version 15.2(7)E7, RELEASE".partition("Version ")[2].partition(",")[0]` vale:
a) `"15.2(7)E7"`  b) `"15.2(7)E7,"`  c) `" RELEASE"`  d) `"Version 15.2(7)E7"`

<details>
<summary><b>Resposta da 1</b></summary>

**a**. O primeiro `partition` fica com o que vem depois de `Version `, e o segundo, com o que vem antes da vírgula.

</details>

**2.** Com `with ConnectHandler(...) as conexao:`, se o `send_command` der erro, a conexão:
a) fica aberta  b) é fechada assim mesmo  c) é refeita  d) trava o script

<details>
<summary><b>Resposta da 2</b></summary>

**b**. Fechar ao sair do bloco, com ou sem erro, é o trabalho do `with`.

</details>

## 🏠 Para casa

- [Lista 14](https://lacouth.github.io/python_telecom-site/listas/lista14/) —
  automação, com testes automáticos no Colab.
- Releia o [capítulo 14 do site](https://lacouth.github.io/python_telecom-site/unidade7-redes/14-automacao/) — o coletor dele é o esqueleto do projeto final.
- **Projeto final:** entrega e apresentação nesta semana. Veja o
  [briefing](https://lacouth.github.io/python_telecom-site/projeto/) e
  [Organizando o projeto em arquivos](https://lacouth.github.io/python_telecom-site/projeto/organizacao/).